# Random Protein Sampling

Generate protein sequence variants by filling masked positions with random amino acids drawn from a codon scheme. Pairs with the masking strategy system to control which positions are mutated.

In [1]:
from proto_tools import (
    RandomProteinSampleConfig,
    RandomProteinSampleInput,
    run_random_protein_sample,
)
from proto_tools.tools.masked_models.masking import MaskingStrategy

## 1. Basic usage with pre-masked sequences

Use `_` to mark positions for random substitution. Unmasked positions are preserved.

In [2]:
inputs = RandomProteinSampleInput(sequences=["MK_AY_AKQR"])
result = run_random_protein_sample(inputs)

print(f"Input:  MK_AY_AKQR")
print(f"Output: {result.sequences[0]}")

Input:  MK_AY_AKQR
Output: MKTAYTAKQR


## 2. Using a masking strategy

Let the masking strategy choose which positions to mutate. Here we mutate exactly 3 random positions.

In [3]:
wild_type = "MKTLAYAKQR"

config = RandomProteinSampleConfig(
    masking_strategy=MaskingStrategy(num_mutations=3),
    seed=42,
)
result = run_random_protein_sample(
    RandomProteinSampleInput(sequences=[wild_type]),
    config,
)

variant = result.sequences[0]
diffs = [i for i, (a, b) in enumerate(zip(wild_type, variant)) if a != b]
print(f"Wild type: {wild_type}")
print(f"Variant:   {variant}")
print(f"Mutated positions: {diffs}")

Wild type: MKTLAYAKQR
Variant:   MKPAGYAKQR
Mutated positions: [2, 3, 4]


## 3. Codon scheme-biased sampling

Different codon schemes bias amino acid frequencies. NNK encodes all 20 amino acids in 32 codons; NDT encodes 12 amino acids in 12 codons with zero redundancy.

In [4]:
for scheme in ["UNIFORM", "NNK", "NDT"]:
    config = RandomProteinSampleConfig(codon_scheme=scheme, seed=42)
    result = run_random_protein_sample(
        RandomProteinSampleInput(sequences=["____"]),
        config,
    )
    print(f"{scheme:>7s}: {result.sequences[0]}")

UNIFORM: PAGF
    NNK: EKSR
    NDT: GNHI


## 4. Batch generation and export

In [5]:
wild_type = "MKTLAYAKQR"
# Duplicate the wild type to generate 5 independent variants
inputs = RandomProteinSampleInput(sequences=[wild_type] * 5)
config = RandomProteinSampleConfig(
    masking_strategy=MaskingStrategy(num_mutations=2),
)
result = run_random_protein_sample(inputs, config)

for i, seq in enumerate(result.sequences):
    print(f"Variant {i}: {seq}")

# Export to FASTA
result.export(name="random_protein_variants", export_path=".", file_format="fasta")

Variant 0: MKTLAYARNR
Variant 1: MITLAYAKQH
Variant 2: MVTLQYAKQR
Variant 3: MKCLAYAKFR
Variant 4: MKCLAYAKQP
